In [1]:
# Cài đặt thư viện kết nối với server của ECMWF
!pip install cdsapi

# Cài đặt các thư viện xử lý dữ liệu mảng và khí tượng (NetCDF)
!pip install xarray[complete] netCDF4 h5netcdf

# Cài đặt dask để hỗ trợ đọc các file dữ liệu lớn mà không làm tràn RAM
!pip install dask

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.3/51.3 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 99.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 70.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.1/89.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.9/151.9 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.6/319.6 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 68.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 108.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 3

In [2]:
import cdsapi
import xarray as xr
import pandas as pd
import numpy as np
from kaggle_secrets import UserSecretsClient

import os
import zipfile
import shutil


In [3]:
import cdsapi
import os
import zipfile
import shutil
from kaggle_secrets import UserSecretsClient

# 1. Thiết lập Key từ Kaggle Secrets
user_secrets = UserSecretsClient()
cds_key = user_secrets.get_secret("key")
cds_url = user_secrets.get_secret("url")

client = cdsapi.Client(url=cds_url, key=cds_key)

In [4]:
years = [str(y) for y in range(2020, 2027)]

# Cấu hình chung cho cả 2 phần
months_all = [f"{i:02d}" for i in range(1, 13)]
days_all = [f"{i:02d}" for i in range(1, 32)]
times_all = ["00:00", "06:00", "12:00", "18:00"]
area_vn = [28.5, 92, -11, 142]

# Chia 12 tháng thành 2 nhóm cho dữ liệu tầng áp suất (Pressure Levels)
month_groups = [
    ["01", "02", "03", "04", "05", "06"],  # Nhóm 1
    ["07", "08", "09", "10", "11", "12"]   # Nhóm 2
]

# =====================================================================
# BẮT ĐẦU VÒNG LẶP QUA TỪNG NĂM
# =====================================================================
for year in years:
    print(f"\n=================================================================")
    print(f"🌍 BẮT ĐẦU XỬ LÝ DỮ LIỆU CHO NĂM {year} 🌍")
    print(f"=================================================================")

    # ĐỊNH NGHĨA 1 FOLDER DUY NHẤT CHO CẢ NĂM
    output_folder = f"/kaggle/working/era5_data/year_{year}"
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        print(f"Đã tạo thư mục lưu trữ chung: {output_folder}")

    # -----------------------------------------------------------------
    # PHẦN 1: TẢI DỮ LIỆU SINGLE LEVELS (SURFACE) -> Lưu vào output_folder
    # -----------------------------------------------------------------
    request_sfc = {
        "product_type": "reanalysis",
        "variable": [
            "10m_u_component_of_wind", 
            "10m_v_component_of_wind",
            "2m_dewpoint_temperature", 
            "2m_temperature",
            "mean_sea_level_pressure", 
            "total_precipitation",
            "geopotential", 
            "land_sea_mask"
        ],
        "year": year,
        "month": months_all,
        "day": days_all,
        "time": times_all,
        "data_format": "netcdf",
        "area": area_vn,
    }

    print(f"\n--- [Năm {year}] Đang yêu cầu dữ liệu Single Levels từ CDS ---")
    try:
        result_sfc = client.retrieve("reanalysis-era5-single-levels", request_sfc)
        remote_path_sfc = result_sfc.location
        is_zip_sfc = remote_path_sfc.endswith('.zip')
        
        temp_download_sfc = os.path.join("/kaggle/working", f"temp_sfc_{year}" + (".zip" if is_zip_sfc else ".nc"))
        print(f"Phát hiện định dạng: {'ZIP' if is_zip_sfc else 'NetCDF'}")
        result_sfc.download(temp_download_sfc)

        if is_zip_sfc:
            print(f"Đang giải nén Single Levels vào folder: {output_folder}...")
            with zipfile.ZipFile(temp_download_sfc, 'r') as zip_ref:
                zip_ref.extractall(output_folder)
            os.remove(temp_download_sfc)
        else:
            final_nc_path = os.path.join(output_folder, f"era5_surface_{year}.nc")
            shutil.move(temp_download_sfc, final_nc_path)
            print(f"Đã di chuyển file vào: {final_nc_path}")
            
    except Exception as e:
        print(f"❌ Có lỗi khi tải Single Levels năm {year}: {e}")

    # -----------------------------------------------------------------
    # PHẦN 2: TẢI DỮ LIỆU PRESSURE LEVELS -> Lưu CÙNG vào output_folder
    # -----------------------------------------------------------------
    for idx, months in enumerate(month_groups):
        print(f"\n--- [Năm {year}] Đang gọi API Pressure lần {idx+1} (Tháng {months[0]} - {months[-1]}) ---")
        
        request_prs = {
            "product_type": "reanalysis",
            "variable": [
                "geopotential",
                "specific_humidity",
                "u_component_of_wind",
                "v_component_of_wind",
            ],
            "year": year,
            "month": months,
            "day": days_all,
            "time": times_all,
            "pressure_level": ["500", "850"],
            "data_format": "netcdf",
            "area": area_vn
        }

        try:
            result_prs = client.retrieve("reanalysis-era5-pressure-levels", request_prs)
            is_zip_prs = result_prs.location.endswith('.zip')
            
            temp_name_prs = f"temp_prs_{year}_part_{idx+1}" + (".zip" if is_zip_prs else ".nc")
            temp_path_prs = os.path.join("/kaggle/working", temp_name_prs)
            
            print(f"Đang tải về: {temp_name_prs}...")
            result_prs.download(temp_path_prs)

            if is_zip_prs:
                print(f"Đang giải nén {temp_name_prs} vào {output_folder}...")
                with zipfile.ZipFile(temp_path_prs, 'r') as zip_ref:
                    zip_ref.extractall(output_folder)
                os.remove(temp_path_prs)
            else:
                final_path_prs = os.path.join(output_folder, f"era5_pressure_{year}_part_{idx+1}.nc")
                shutil.move(temp_path_prs, final_path_prs)
                print(f"Đã di chuyển file vào: {final_path_prs}")

        except Exception as e:
            print(f"❌ Có lỗi xảy ra ở tầng áp suất năm {year} (lần call {idx+1}): {e}")

print("\n🚀 --- HOÀN THÀNH TOÀN BỘ TIẾN TRÌNH --- 🚀")


🌍 BẮT ĐẦU XỬ LÝ DỮ LIỆU CHO NĂM 2020 🌍
Đã tạo thư mục lưu trữ chung: /kaggle/working/era5_data/year_2020

--- [Năm 2020] Đang yêu cầu dữ liệu Single Levels từ CDS ---


2026-06-01 11:12:58,594 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-06-01 11:12:58,595 INFO Request ID is 4c9b9934-1681-476d-9c23-5648321789a4
2026-06-01 11:12:58,724 INFO status has been updated to accepted
Recovering from HTTP error [500 Internal Server Error], attempt 1 of 500
Retrying in 120 seconds
2026-06-01 11:55:56,040 INFO status has been updated to running
2026-06-01 12:06:02,732 INFO status has been updated to successful


Phát hiện định dạng: ZIP


da0408b0e02e6ae382608bafcccadf25.zip:   0%|          | 0.00/483M [00:00<?, ?B/s]

Đang giải nén Single Levels vào folder: /kaggle/working/era5_data/year_2020...

--- [Năm 2020] Đang gọi API Pressure lần 1 (Tháng 01 - 06) ---


...

Đang giải nén Single Levels vào folder: /kaggle/working/era5_data/year_2026...

--- [Năm 2026] Đang gọi API Pressure lần 1 (Tháng 01 - 06) ---


2026-06-01 18:38:22,262 INFO Request ID is a2df2720-ef37-4db6-ad5d-7b3d4a1204dd
2026-06-01 18:38:22,395 INFO status has been updated to accepted
2026-06-01 19:10:56,934 INFO status has been updated to running
2026-06-01 19:14:57,947 INFO status has been updated to successful


Đang tải về: temp_prs_2026_part_1.nc...


9bb7f9d0ca02076fe774940e63303204.nc:   0%|          | 0.00/265M [00:00<?, ?B/s]

Đã di chuyển file vào: /kaggle/working/era5_data/year_2026/era5_pressure_2026_part_1.nc

--- [Năm 2026] Đang gọi API Pressure lần 2 (Tháng 07 - 12) ---
❌ Có lỗi xảy ra ở tầng áp suất năm 2026 (lần call 2): 400 Client Error: Bad Request for url: https://cds.climate.copernicus.eu/api/retrieve/v1/processes/reanalysis-era5-pressure-levels/execution
invalid request
None of the data you have requested is available yet, please revise the period requested. The latest date available for this dataset is: 2026-05-27 19:00

🚀 --- HOÀN THÀNH TOÀN BỘ TIẾN TRÌNH --- 🚀
